In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 35.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 77.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/unified_obstacles /content/

Mounted at /content/drive


In [5]:
import os
import shutil
import random

TRAIN_IMG = '/content/unified_obstacles/train/images'
TRAIN_LBL = '/content/unified_obstacles/train/labels'
TARGET = 500

# Количество объектов каждого класса до
class_counts = {i:0 for i in range(8)}
for lbl in os.listdir(TRAIN_LBL):
    if not lbl.endswith('.txt'): continue
    with open(os.path.join(TRAIN_LBL, lbl)) as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts[cls] += 1

print("До оверсэмплинга:", class_counts)

# Создаём копии
for cls, cnt in class_counts.items():
    if cnt >= TARGET:
        print(f"Класс {cls}: уже {cnt}, пропускаем.")
        continue

    # Все изображения, в которых присутствует этот класс
    candidates = []
    for lbl in os.listdir(TRAIN_LBL):
        if not lbl.endswith('.txt'): continue
        base = os.path.splitext(lbl)[0]
        img_ext = None
        for ext in ['.jpg','.jpeg','.png']:
            if os.path.exists(os.path.join(TRAIN_IMG, base+ext)):
                img_ext = ext
                break
        if img_ext is None: continue
        with open(os.path.join(TRAIN_LBL, lbl)) as f:
            classes_present = {int(line.split()[0]) for line in f}
        if cls in classes_present:
            candidates.append((base+img_ext, lbl))

    if not candidates:
        print(f"Класс {cls}: нет подходящих изображений!")
        continue

    needed = TARGET - cnt
    for i in range(needed):
        img, lbl = random.choice(candidates)
        base, ext = os.path.splitext(img)
        new_img = f"{base}_ovs{i}{ext}"
        new_lbl = f"{base}_ovs{i}.txt"
        shutil.copy2(os.path.join(TRAIN_IMG, img), os.path.join(TRAIN_IMG, new_img))
        shutil.copy2(os.path.join(TRAIN_LBL, lbl), os.path.join(TRAIN_LBL, new_lbl))

# Пересчитаем после оверсэмплинга
class_counts_after = {i:0 for i in range(8)}
for lbl in os.listdir(TRAIN_LBL):
    if not lbl.endswith('.txt'): continue
    with open(os.path.join(TRAIN_LBL, lbl)) as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts_after[cls] += 1

print("После оверсэмплинга:", class_counts_after)

До оверсэмплинга: {0: 889, 1: 122, 2: 869, 3: 363, 4: 397, 5: 292, 6: 329, 7: 288}
Класс 0: уже 889, пропускаем.
Класс 2: уже 869, пропускаем.
После оверсэмплинга: {0: 889, 1: 502, 2: 869, 3: 564, 4: 576, 5: 630, 6: 527, 7: 521}


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')

model.train(
    data='/content/unified_obstacles/data.yaml',
    epochs=70,
    patience=7,
    imgsz=640,
    batch=64,
    device=0,
    project='/content/drive/MyDrive/obstacles_training',
    name='run_final'
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.7.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/unified_obstacles/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_final-10, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.4 MB/s eta 0:00:00


In [5]:
!cp -r /content/drive/MyDrive/unified_obstacles /content/

In [6]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/obstacles_training/run_final-10/weights/best.pt')
metrics = model.val()

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1534.7±909.3 MB/s, size: 180.9 KB)
val: Scanning /content/unified_obstacles/valid/labels... 580 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 580/580 1.3Kit/s 0.4s
val: New cache created: /content/unified_obstacles/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 3.8s/it 2:20
                   all        580        926      0.782      0.713      0.755      0.426
                animal        120        224      0.583      0.451       0.47       0.27
          open_manhole         31         32      0.957      0.969      0.957      0.602
       waste_container        120        243      0.794      0.728      0.799      0.496
               pothole         59        106      0.823     

In [7]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/obstacles_training/run_final-10/weights/best.pt')
model.export(format='tflite', imgsz=640, int8=True)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ INT8 export requires a missing 'data' arg for calibration. Using default 'data=coco8.yaml'.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/obstacles_training/run_final-10/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (5.2 MB)
TensorFlow SavedModel: collecting INT8 calibration images from 'data=coco8.yaml'

WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 1.9Kfiles/s 0.0s
Dataset download success ✅ (0.2s), saved to /content/datasets

Fast image access ✅ (ping: 0.0±0.0 ms, read: 1202.6±53

'/content/drive/MyDrive/obstacles_training/run_final-10/weights/best_saved_model/best_int8.tflite'